# Actividad 2: Luces nocturnas y `config.py`

En este cuaderno pasan dos cosas.

**El satélite.** El producto Black Marble de la NASA mide la luz artificial nocturna con
una resolución de 500 m, todas las noches, desde 2012, en todo el planeta. Es un indicador
de actividad económica que llega con cerca de una semana de rezago y con tanto detalle
geográfico como se quiera.

**El archivo de configuración.** Esta es la primera actividad que lee `config.py`.
Hasta ahora cada cuaderno era autosuficiente: el nombre de los datos o de los archivos de
salida estaba escrito en el propio código. El archivo de configuración reúne todo lo que
se puede cambiar en un solo lugar.

La regla es:

> **Se edita `config.py`. No se edita el cuaderno.**

Al final de la actividad se podrá apuntar el mismo código, sin modificarlo, a otro país y
a otro periodo, y va a funcionar.

---

## Entorno `geo`

Este cuaderno necesita GDAL, HDF5 y librerías geoespaciales que no están en el entorno
principal. Hay que correrlo con el kernel **`geo`**.

## La carga de la configuración

Esta celda es igual en todos los cuadernos que usan `config.py`. La celda:

1. busca `config.py`;
2. lo carga como un módulo llamado `cfg`, de modo que cada parámetro queda disponible
   como `cfg.COUNTRY`, `cfg.TILES`, etc.;
3. arma las rutas de las carpetas estándar en `P`;
4. importa las funciones de `ntl_helpers.py`, que hacen el trabajo pesado y mantienen
   este cuaderno corto.

Leer lo que imprime: todo sale de `config.py`.

In [ ]:
# ===========================================================================
# CONFIG LOADER  --  edit config.py, NOT this cell.
# Gives you `cfg` (every country knob) and `P` (the standard paths).
# ===========================================================================
import os, sys, importlib, importlib.util
import numpy as np
from tqdm import tqdm

# config.py and ntl_helpers.py live beside this notebook.
CODE_DIR = os.getcwd()
if not os.path.exists(os.path.join(CODE_DIR, "config.py")):
    CODE_DIR = os.path.abspath(os.path.join(CODE_DIR, "..", "activity"))

_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import ntl_helpers
importlib.reload(ntl_helpers)      # pick up edits without restarting the kernel
from ntl_helpers import *

for _k in ("raw", "data", "ntl_tmp"):
    os.makedirs(P[_k], exist_ok=True)

# Can GDAL actually READ an .h5? Checked here, before anything is downloaded,
# because without the HDF5 driver nothing raises -- every tile just reads as
# empty and the run ends "fine" with no data. Raises with the fix if not.
check_hdf5_driver()

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE}/{cfg.GADM_ISO3})")
print(f"         tiles      : {cfg.TILES}   (download: {cfg.download_tiles()})")
print(f"         window     : {cfg.NTL_DATE_RANGE or 'ALL available dates'}")
print(f"         outputs    : raw/{cfg.NTL_SHAPE_CSV}, raw/{cfg.NTL_OUTPUT_CSV}")
print(f"         tile store : {P['nl_root']}")
print(f"         HDF5 driver: OK")


## Archivo de configuración

Abrir `config.py`, que está en la misma carpeta que este cuaderno.

Dos parámetros controlan todo lo que sigue:

```python
COUNTRY        = "Bolivia"
NTL_DATE_RANGE = "2026-01-01 2026-01-09"
```

`COUNTRY` tiene que ser una de las claves del registro `COUNTRIES` de ese archivo.

La celda siguiente imprime el registro: qué países hay y qué resulta para cada uno.

In [ ]:
import pandas as pd

registry = pd.DataFrame([
    {"country": name,
     "code":    v["iso2"],
     "gadm":    v["iso3"],
     "tiles":   " ".join(v["tiles"]),
     "n_tiles": len(v["tiles"])}
    for name, v in cfg.COUNTRIES.items()
]).set_index("country").sort_index()

print("Currently selected:", cfg.COUNTRY, "\n")
registry

## El token de la NASA

El archivo de imágenes es gratuito, pero no anónimo. Hace falta una cuenta de Earthdata y
un token propio:

1. Registrarse en <https://urs.earthdata.nasa.gov>
2. Entrar a <https://ladsweb.modaps.eosdis.nasa.gov> → **My Account → Generate Token**
3. Pegar el texto en `NASA_TOKEN`, en `config.py`

Los tokens vencen, normalmente a los 60 días. Si las descargas empiezan a devolver `401`,
casi siempre es eso: generar uno nuevo.

Si el token está vacío, la celda siguiente no descarga nada. Explica qué hacer y sigue,
así que el resto del cuaderno corre con las imágenes que ya estén en el disco.

In [ ]:
api_key   = cfg.NASA_TOKEN
HAVE_TOKEN = bool(api_key) and not api_key.startswith("PASTE")

# What is already on disk, tile by tile?
inventory = {t: len(list_tile_rasters(P["nl_root"], [t], make=True)) for t in cfg.TILES}
print("Tiles on disk:")
for t, n in inventory.items():
    print(f"   {t}: {n:,} file(s)")

if not HAVE_TOKEN:
    print("\nNASA_TOKEN is empty in config.py -> skipping download.")
    print("Set it to run the download cell below; the rest of the notebook will")
    print("still work on the files already present.")

## Descarga

`missing_targets` le pregunta al servidor de LAADS qué días existen para esas piezas y ese
periodo, lo compara con lo que ya está en el disco y devuelve solo la diferencia.

`download_many` descarga los archivos en paralelo, con reintentos. Escribe primero un
archivo `.part` y lo renombra al terminar, así que una corrida interrumpida nunca deja una
imagen a medias.

Las dos están en `ntl_helpers.py`. El cuaderno solo las llama.

In [ ]:
from datetime import datetime

# The window comes from cfg.ntl_window(), NOT from cfg.NTL_DATE_RANGE directly:
# ntl_window() applies NTL_FIRST_DATE as a floor even when the range is None, so
# "I forgot to set a window" can no longer mean "fetch everything since 2012".
start_date, end_date = cfg.ntl_window()

_tiles = len(cfg.download_tiles())
if end_date is not None:
    _days  = (end_date - start_date).days + 1
    _files = _days * _tiles
    print(f"Window: {start_date.date()} -> {end_date.date()}  ({_days} days)")
    print(f"At most {_files:,} files ({_days} days x {_tiles} tiles), "
          f"about {_files * 25 / 1024:.1f} GB if none are on disk yet.")
else:
    print(f"Window: {start_date.date()} -> latest available (open end).")
    print(f"Open end, so the size depends on today's date. The run is capped at "
          f"cfg.NTL_MAX_FILES = {cfg.NTL_MAX_FILES} files and floored at "
          f"cfg.NTL_FIRST_DATE = {cfg.NTL_FIRST_DATE}.")

# The download log lives with the tiles it describes: hfiles/ is local-only.
log_path = os.path.join(P["nl_root"], "ntl_download_log.txt")

if HAVE_TOKEN:
    target = missing_targets(cfg.download_tiles(), api_key, P["nl_root"],
                             start_date=start_date, end_date=end_date)
    print()
    print(f"{len(target)} file(s) available on the server but missing on disk "
          f"(~{len(target) * 25 / 1024:.1f} GB to fetch).")
    if target:
        # Per tile, so a tile that is quietly failing is visible immediately.
        from collections import Counter
        for _t, _n in sorted(Counter(t for t, _, _ in target).items()):
            print(f"   {_t}: {_n:,}")
        try:
            result = download_many(target, api_key, P["nl_root"],
                                   log_path=log_path, max_workers=8,
                                   max_files=cfg.NTL_MAX_FILES)
        except ntl_helpers.TooManyTargets as e:
            # Deliberate stop, not a crash: nothing has been downloaded.
            print()
            print("DETENIDO --", e)
            result = {}
        missing = result.get("unavailable", 0) + result.get("failed", 0)
        if result:
            print("All files downloaded." if missing == 0
                  else f"{missing} file(s) not downloaded -- see {log_path}")
    else:
        print("Nothing missing: up to date.")
else:
    print()
    print("No token -> skipped. Working with what is already on disk.")

rasterFiles = list_tile_rasters(P["nl_root"], list(cfg.TILES), make=False)
print()
print(f"{len(rasterFiles):,} raster file(s) available across {cfg.TILES}")


# Medidas de Luces

## Paso 1: Uso de las fronteras/contorno

El contorno del país sale de **GADM**, la base abierta de límites administrativos más
usada. `cfg.GADM_ZIP` se arma con el código ISO3 del registro.

In [ ]:
import geopandas as gpd
import urllib.request

# The GADM boundary is ~16 MB and never changes between runs, so it is cached in
# code/shapes/ and fetched only the first time. That also means the notebook
# runs offline afterwards -- useful in a workshop room with hostile wifi.
shape_dir = os.path.join(CODE_DIR, "shapes")
os.makedirs(shape_dir, exist_ok=True)
shape_zip = os.path.join(shape_dir, cfg.GADM_ZIP)
shape_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/" + cfg.GADM_ZIP

if not os.path.exists(shape_zip):
    print("Downloading boundary (once):", shape_url)
    urllib.request.urlretrieve(shape_url, shape_zip)
print("Boundary:", shape_zip)

# Layer _0 is the national outline. The deeper layers (_1 departments, _2
# provinces, _3 municipalities) are the same border with more vertices -- for
# Bolivia, _3 is 340 polygons that dissolve back to exactly this one.
gdf         = gpd.read_file(f"zip://{shape_zip}!gadm41_{cfg.GADM_ISO3}_0.shp")
gdf_country = gdf[gdf.geometry.notnull()].dissolve().to_crs("EPSG:4326")

print(f"{cfg.COUNTRY}: {len(gdf)} administrative unit(s) dissolved into one polygon")
print(f"bounds: {tuple(round(b, 2) for b in gdf_country.total_bounds)}")

gdf_country.explore(tiles="CartoDB dark_matter")


## Paso 2: dos preguntas, dos funciones

| Pregunta | Función | Qué recibe |
|---|---|---|
| ¿Cuánta luz tiene esta **zona**? | `processHD5_shape` | un polígono |
| ¿Cuánta luz tiene este **sitio**? | `processHD5` | una lista de puntos lat/lon |

Las dos están en `ntl_helpers.py`.

In [ ]:
# Same window as the download cell, floor included: ntl_window() applies
# NTL_FIRST_DATE even when NTL_DATE_RANGE is None, so the extraction step can
# never widen beyond what the download step was allowed to fetch.
_ws, _we = cfg.ntl_window()

shape_csv = os.path.join(P["raw"], cfg.NTL_SHAPE_CSV)
files = rasters_in_range(P["nl_root"], list(cfg.TILES), start=_ws, end=_we)

# RESUME. Dates already complete in raw/<NTL_SHAPE_CSV> are filtered out here,
# on the date in the filename, so those .h5 files are never opened. That is
# what makes "extract the series, then evict the tiles to the cloud" safe:
# reading an evicted file would drag it back down again.
# A date counts as complete only when all len(cfg.TILES) tiles reported.
done    = shape_dates_done(shape_csv, len(cfg.TILES))
pending = pending_rasters(files, done, label="Country NTL")

recs = []
for _path, _dt in tqdm(pending, desc="Country NTL"):
    try:
        recs.append(processHD5_shape(_path, 2, P["ntl_tmp"], gdf_country, _dt))
    except Exception as e:
        print(f"  skip {os.path.basename(_path)}: {type(e).__name__}: {e}")

# Merges the new dates into whatever the CSV already held, writes it back, and
# returns the WHOLE series -- so the plots below work even on a run that read
# nothing. Across tiles the totals add and the mean is recomputed as sum/n,
# never averaged across tile means (see update_shape_csv).
shape_df = update_shape_csv(recs, shape_csv, len(cfg.TILES))

if not len(shape_df):
    raise RuntimeError(
        "No data. hfiles/<tile>/ is empty for this window -- run the download "
        "cell -- or cfg.NTL_DATE_RANGE points at dates never downloaded.")

# A date is comparable to another only if the same tiles reported on both.
_n = shape_df["ntl_n"]
print(f"pixels per date: median {_n.median():,.0f}  min {_n.min():,.0f}  max {_n.max():,.0f}")
_partial = shape_df.index[shape_df["ntl_tiles"] < len(cfg.TILES)]
if len(_partial):
    print(f"WARNING: {len(_partial)} date(s) are missing a tile and will be RE-READ "
          f"on the next run once it is downloaded: {[str(d.date()) for d in _partial[:8]]}")

print()
print(f"{len(shape_df)} row(s) in raw/{cfg.NTL_SHAPE_CSV}  ({len(recs)} file(s) read this run)")
shape_df.tail()


## Paso 3: Serie de tiempo diaria

In [ ]:
import matplotlib.pyplot as plt

win_start, win_end = shape_df.index.min(), shape_df.index.max()

if cfg.NTL_EVENT_DATE:
    marks = [(pd.Timestamp(cfg.NTL_EVENT_DATE), cfg.NTL_EVENT_LABEL or "event")]
else:
    marks = [(win_start, "start"), (win_end, "end")]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(shape_df.index, shape_df["ntl_mean"], color="#005BAC", linewidth=1.8,
        marker="o", markersize=3, label="NTL mean (nW/cm2/sr)")
for _d, _lab in marks:
    ax.axvline(_d, color="#CC0000", linewidth=1.5, linestyle="--", zorder=3)
    ax.text(_d, ax.get_ylim()[1], f" {_lab}", color="#CC0000", fontsize=9,
            va="top", ha="left")

ax.set_title(f"{cfg.COUNTRY} - nighttime lights, country mean",
             fontsize=13, weight="bold")
ax.set_ylabel("nW/cm2/sr")
ax.grid(axis="y", linestyle=":", alpha=0.5)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
print("saved ->", save_fig(fig, P, "ntl_country_mean"))
plt.show()


## Paso 4: Mapa (antes y después)

Las mismas dos fechas, dibujadas como imágenes. Con un evento definido se muestran
`evento ± NTL_EVENT_WINDOW` días; sin evento, el primer y el último día del periodo.

**Cómo leer el mapa.** La escala de colores va de 0 a ~0,3 nW/cm²/sr: todo el azul y el
violeta es **oscuridad**. Las ciudades superan los 5 nW y salen amarillas en ambas fechas.
La diferencia entre azul y violeta en el campo son centésimas de nanovatio, y viene sobre
todo de las **nubes**: donde estaba nublado, la imagen no se observa esa noche y se rellena
con la última noche despejada (la capa *gap-filled*). Una noche nublada se ve más violeta,
no porque haya más luz.

In [ ]:
import contextily as cx
import colorcet as cc

if cfg.NTL_EVENT_DATE:
    _ev = pd.Timestamp(cfg.NTL_EVENT_DATE)
    d_before = _ev - pd.Timedelta(days=cfg.NTL_EVENT_WINDOW)
    d_after  = _ev + pd.Timedelta(days=cfg.NTL_EVENT_WINDOW)
    _lab = cfg.NTL_EVENT_LABEL or "event"
    lab_before = f"{cfg.NTL_EVENT_WINDOW} days before {_lab}"
    lab_after  = f"{cfg.NTL_EVENT_WINDOW} days after {_lab}"
else:
    d_before, d_after = shape_df.index.min(), shape_df.index.max()
    lab_before, lab_after = "window start", "window end"

# Set to False to draw the rasters on a plain background and make the cell
# fully offline -- a basemap is fetched from a public tile server.
BASEMAP = True

# ALL tiles for each date, not just the first. A country that straddles a tile
# boundary is drawn from the mosaic of all of them; take only the first and you
# map whichever 10-degree square sorts first -- for Bolivia that is h11v09, a
# square of Amazon holding 0.3% of the country.
# The maps are the ONLY step that still opens a tile once the series is
# extracted -- they need pixels, which no CSV holds. Turn them off in
# config.py before running over a tile store that lives in the cloud.
DRAW_MAPS = getattr(cfg, "NTL_DRAW_MAPS", True)

f_before = find_h5s_for_date(d_before, P["nl_root"], list(cfg.TILES)) if DRAW_MAPS else []
f_after  = find_h5s_for_date(d_after,  P["nl_root"], list(cfg.TILES)) if DRAW_MAPS else []
_tiles_of = lambda fs: " + ".join(os.path.basename(f).split(".")[2] for f in fs) or "MISSING"
if DRAW_MAPS:
    print(f"before ({d_before.date()}): {len(f_before)}/{len(cfg.TILES)} tile(s)  {_tiles_of(f_before)}")
    print(f"after  ({d_after.date()}): {len(f_after)}/{len(cfg.TILES)} tile(s)  {_tiles_of(f_after)}")

# The maps are the one step no CSV can satisfy: a picture needs the pixels,
# so these two dates are read from the .h5 files even when the series is
# fully extracted. If the tiles have been evicted to cloud storage, reading
# them here is what pulls them back -- and if they are simply gone, this is
# where you find out. Either way the cell reports it and the notebook goes on.
_maps_ok = DRAW_MAPS and bool(f_before) and bool(f_after)
if _maps_ok:
    try:
        arr_b, lons, lats = raster_to_array(f_before, 2, gdf_country, P["ntl_tmp"])
        arr_a, _, _       = raster_to_array(f_after,  2, gdf_country, P["ntl_tmp"])
    except Exception as e:
        _maps_ok = False
        print(f"\nCould not READ the tiles for those dates ({type(e).__name__}).")
        print("find_h5s_for_date matched them by filename, so the names are on disk,")
        print("but the contents are not usable. That is what an evicted/cloud-only or")
        print("truncated .h5 looks like. The series above is unaffected -- it came from")
        print("the CSV. To draw the maps, re-download just these two dates, or point")
        print("NTL_EVENT_DATE / NTL_DATE_RANGE at dates whose tiles are still local.")

if not _maps_ok:
    print("cfg.NTL_DRAW_MAPS is False -> maps skipped, no tile opened."
          if not DRAW_MAPS else "Skipping the before/after maps.")
else:

    both = np.concatenate([arr_b[~np.isnan(arr_b)], arr_a[~np.isnan(arr_a)]])
    vmin, vmax = np.nanpercentile(both, 2), np.nanpercentile(both, 98)

    # imshow on the regular grid, NOT pcolormesh. Both draw the same picture,
    # but pcolormesh builds one quadrilateral per pixel, and Bolivia clipped out
    # of a four-tile mosaic is ~9 million of them -- minutes of rendering and a
    # lot of memory. imshow blits the array. The Caribbean countries this
    # notebook came from are small enough that the difference never showed.
    dx, dy = abs(lons[1] - lons[0]), abs(lats[1] - lats[0])
    extent = [lons[0] - dx / 2, lons[-1] + dx / 2, lats[-1] - dy / 2, lats[0] + dy / 2]
    print(f"grid: {arr_b.shape[0]} x {arr_b.shape[1]} pixels")

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    fig.subplots_adjust(bottom=0.15)
    titles = [f"{lab_before}\n({d_before.strftime('%b %d, %Y')})",
              f"{lab_after}\n({d_after.strftime('%b %d, %Y')})"]
    for ax, arr, title in zip(axes, [arr_b, arr_a], titles):
        # EXPLICIT zorder, on every layer. add_basemap() draws AFTER this call,
        # and at equal zorder the later artist wins -- which hid the lights
        # under the basemap entirely. Pixels outside Bolivia are NaN and so
        # render transparent, which is how the basemap shows around the country.
        im = ax.imshow(arr, cmap=cc.cm.bmy, vmin=vmin, vmax=vmax,
                       extent=extent, origin="upper", interpolation="nearest",
                       zorder=2)
        gdf_country.boundary.plot(ax=ax, color="#444444", linewidth=0.8, zorder=3)
        ax.set_title(title, fontsize=13, weight="bold")
        if BASEMAP:
            try:
                # Esri, not CartoDB and not OpenStreetMap. Carto now refuses
                # unkeyed requests and serves tiles stamped "API KEY REQUIRED";
                # OSM's volunteer servers answer contextily's default user
                # agent with a 403 "Access blocked" tile at this zoom. Neither
                # raises -- both just paint their refusal onto the figure.
                # Esri's gray canvas needs no key and stays out of the way.
                cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas,
                               crs="EPSG:4326", attribution_size=6, zorder=0)
            except Exception as e:
                print(f"  basemap unavailable ({type(e).__name__}) -- plotting without it")
        ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
        ax.set_aspect("equal")
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

    cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.025])
    fig.colorbar(im, cax=cbar_ax, orientation="horizontal",
                 label="NTL radiance (nW/cm2/sr)")
    fig.suptitle(f"Nighttime light radiance - {cfg.COUNTRY}", fontsize=15, weight="bold")
    fig.text(0.01, 0.01, "Source: NASA Black Marble VNP46A2", fontsize=10)
    print("saved ->", save_fig(fig, P, "ntl_map_before_after"))
    plt.show()


## Paso 5: Coordenadas Geograficas

El promedio del país es un instrumento grueso. Un aeropuerto, una mina, una planta de gas
o un centro comercial tienen cada uno su serie, y son esas series las que se conectan con
los sectores que se quiere predecir.

`raw/Coordinates.xlsx` tiene la lista de sitios: sector, nombre, ciudad, latitud y
longitud. `processHD5` devuelve dos lecturas por sitio y por noche:

* `DNBvalue1`: el píxel que contiene el punto
* `DNBvalue3`: el promedio del bloque de 3×3 píxeles alrededor

En la práctica conviene usar `DNBvalue3`. Un píxel de VIIRS mide unos 464 m y la
geolocalización no es perfecta, así que un solo píxel puede errar el sitio por una celda;
el promedio de 3×3 es mucho más estable. La contracara es que sitios a menos de ~464 m
entre sí comparten píxeles y no son series realmente independientes.

In [ ]:
coord_path = os.path.join(P["raw"], cfg.COORDINATES_XLSX)
site_csv   = os.path.join(P["raw"], cfg.NTL_OUTPUT_CSV)

if not os.path.exists(coord_path):
    print(f"No {cfg.COORDINATES_XLSX} in raw/ -- skipping the site-level extraction.")
    site_df = pd.DataFrame()
else:
    locations = pd.read_excel(coord_path)

    # Strip accents and stray whitespace so names stay stable as merge keys --
    # and so they still match the names already written into the CSV.
    for col in ("location", "city", "sector"):
        locations[col] = (locations[col].fillna("").astype(str)
                            .str.normalize("NFKD")
                            .str.encode("ascii", errors="ignore").str.decode("utf-8")
                            .str.strip())

    coords = locations[["sector", "location", "city", "lat", "lon"]].to_dict("records")
    print(f"{len(coords)} site(s) in {cfg.COORDINATES_XLSX}")
    print("sectors:", ", ".join(sorted(locations["sector"].unique())))
    print()

    # RESUME, same idea as the country series but judged per SITE: a date is
    # done only when the CSV already holds a row for every location requested
    # here. Add a site to Coordinates.xlsx and every date falls short again,
    # so the new site is backfilled across the whole history automatically.
    done    = site_dates_done(site_csv, locations["location"])
    pending = pending_rasters(files, done, label="Site NTL")

    rows = []
    for _path, _dt in tqdm(pending, desc="Site NTL"):
        try:
            rows.extend(processHD5(_path, 2, P["ntl_tmp"], coords, _dt))
        except Exception as e:
            print(f"  skip {os.path.basename(_path)}: {type(e).__name__}: {e}")

    site_df = update_site_csv(rows, site_csv)
    print()
    print(f"{len(site_df):,} site-day(s) in raw/{cfg.NTL_OUTPUT_CSV}  "
          f"({len(rows):,} written this run)")

site_df.head()


In [ ]:
# Mean brightness by sector over the window -- the site data in one picture.
if len(site_df):
    by_sector = (site_df.groupby(["date", "sector"])["DNBvalue3"]
                        .mean().unstack("sector"))

    fig, ax = plt.subplots(figsize=(12, 4))
    by_sector.plot(ax=ax, linewidth=1.6, marker="o", markersize=3)
    ax.set_title(f"{cfg.COUNTRY} - mean site radiance by sector", fontsize=13, weight="bold")
    ax.set_ylabel("nW/cm2/sr")
    ax.set_xlabel("")
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.legend(frameon=False, ncol=3, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    print("saved ->", save_fig(fig, P, "ntl_by_sector"))
    plt.show()

    display(by_sector.describe().T[["count", "mean", "std", "min", "max"]].round(2))
